In [44]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv('GOOGLE_API_KEY')

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.5)

In [45]:
pizzeria_menu = """
    --- Pizzeria Menu ---
    Regular Menu:
    - Margherita: $8.50
    - Pepperoni: $9.50
    - Vegetarian: $9.00
    - BBQ Chicken: $10.00
    
    Special Lunch Menu (Available everyday, 11:00 - 14:00):
    - Truffle Mushroom Pizza: $7.50
    - Spicy Salami Pizza: $7.00
    - Pesto Chicken Pizza: $7.50
    """

### Tool/Function calling to get weather and traffic

In [46]:
from tools import get_weather, get_traffic, GetWeather, GetTraffic

### Customer Agents

In [47]:
# create customer ordering agent
# pydantic for structured output produced by llm order agent
from pydantic import BaseModel, Field

class OrderOutput(BaseModel):
    """Structured Output of Customer Order"""
    action: str = Field(description="Action taken by customer")
    pizza_type: str = Field(description="The kind of pizza the customer want to order, leave empty if customer does not order")
    timestamp: str = Field(description='The timestamp at which the activity is finished')

from langchain_core.messages import HumanMessage, ToolMessage, AIMessage

def customer_order_agent(customer, start_time, end_time, call_tools = False):
    name = customer['name']
    distance = customer['distance']
    character = customer['character']

    order_agent_prompt = """
    You are {name}, living {distance} km from Da Pipo Pizzeria. {character}

    Do you want to order from Da Pipo Pizzeria?

    Here are the menu: 
    {pizzeria_menu}

    Your decision to order depends on multiple factors:  
    - **Weather** (e.g., rainy weather makes it a great time for pizza but may delay delivery)  
    - **Time** (e.g., you're lazier to cook on weekends)  
    - **Hunger level** (whether you're hungry right now)  
    - **Traffic** (heavy traffic may slow delivery)  
    - **Distance** (closer distance means faster delivery)  

    The current time is {start_time}. You have until {end_time} to make your decision. Based on these factors, decide whether to order pizza or not within the next 10 minutes. 
    It takes you sometime to decide and you don't have to decide right away at {start_time}.

    Please respond with:
    - **action**: "place order" if you want to order pizza within the time frame, or "next time" if not.
    - **pizza type**: The pizza you want (only if placing an order).
    - **timestamp**: The exact time of your decision (within the time frame).  
    """

    prompt = order_agent_prompt.format(name=name,
                                    distance=distance,
                                    character=character,
                                    start_time=start_time,
                                    end_time=end_time,
                                    pizzeria_menu=pizzeria_menu)

    # initializaing messages with human prompt
    messages = [HumanMessage(prompt)]

    if call_tools:
        # Step 1: Use LLM with tools to determine if tool calls are needed
        llm_with_tools = llm.bind_tools([GetWeather, GetTraffic])
        ai_response = llm_with_tools.invoke(messages)

        if ai_response.tool_calls:
            print('Oder customer calling tools')

            if ai_response.content:
                messages.append(ai_response)

            # Step 2: Execute each tool and append its response
            tool_map = {'getweather': get_weather, 'gettraffic': get_traffic}
            for tool_call in ai_response.tool_calls:
                tool_fn = tool_map[tool_call['name'].lower()]
                tool_result = tool_fn.invoke(tool_call['args'])
                messages.append(ToolMessage(tool_result, tool_call_id=tool_call['id']))

            # Step 3: Re-invoke LLM with all messages to generate final response
            final_response = llm_with_tools.invoke(messages)
            if final_response.content:
                messages.append(final_response)
            else:
                print('Unstructured output is empty')

        else:
            print('Oder customer does not call tools')
            ai_response = llm.invoke(messages)
            if ai_response.content:
                messages.append(ai_response)
    else:
        # No tools used, simple LLM call
        ai_response = llm.invoke(messages)
        if ai_response.content:
            messages.append(ai_response)

    # Step 4: Extract structured output from full conversation
    structured_llm = llm.with_structured_output(OrderOutput)
    output = structured_llm.invoke(messages)

    return output

In [48]:
# create customer cancelling agent
# pydantic for structured output produced by llm order agent
class CancelOutput(BaseModel):
    """Structured Output of Customer Order"""
    action: str = Field(description="Action taken by customer")
    timestamp: str = Field(description='The timestamp at which the activity is finished')

from langchain_core.prompts import PromptTemplate

def customer_cancel_agent(customer, order, start_time, end_time):
    name = customer['name']
    distance = customer['distance']
    character = customer['character']

    pizza_type = order['pizza_type']
    ordering_timestamp = order['timestamps'][0]

    customer_cancel_prompt = """
    You are {name}, living {distance} km from Da Pipo Pizzeria. {character}

    You recently placed an order for **{pizza_type}** at **{ordering_timestamp}**. The order has been registered but not yet confirmed.

    You should only **cancel the order** for special reasons, such as:
    - **A sudden change in plans** (e.g., needing to leave home unexpectedly).
    - **A mistake in your order** (e.g., ordering the wrong pizza type).

    Otherwise, you should wait for your order.

    You have 10 minutes from {start_time} to {end_time} to decide. Please provide the following:

    - **action**: "cancel order" if you wish to cancel, or "wait for order" if you prefer to wait.
    - **timestamp**: The exact time of your decision (within the time frame).

    Please choose your action.
    """

    prompt = PromptTemplate.from_template(customer_cancel_prompt)
    structured_llm = llm.with_structured_output(CancelOutput)
    customer_cancel_agent = prompt| structured_llm
    output = customer_cancel_agent.invoke({'name': name,
                                           'distance': distance,
                                           'character': character,
                                           'pizza_type': pizza_type,
                                           'ordering_timestamp': ordering_timestamp,
                                           'start_time': start_time,
                                           'end_time': end_time})
    return output

### Pizzeria Agent

In [49]:
from typing import List
from pydantic import BaseModel, Field

# get order latest state from system state
from helpers import get_latest_states

# pydatic for structured output from llm pizzeria agent
class PizzeriaOutputSingle(BaseModel):
    """Structured Output of System Agent"""
    order_id: str = Field(description="Order ID")
    action: str = Field(description="Action taken by the process participant")
    timestamp: str = Field(description="The timestamp at which the action is taken place")

class PizzeriaOutputMultiple(BaseModel):
    multiple_actions: List[PizzeriaOutputSingle] = Field(description="Multiple actions taken by the process participant")

def pizzeria_agent(system_state, start_time, end_time, call_tools=False):
# prompt
    pizzeria_prompt = """
    You are the owner and sole worker of Da Pipo Pizzeria, operating exclusively through online orders. Your pezzeria is open from 8:00 to 22:00 daily.

    When a new order is placed, you must manage the following steps to fulfill it efficiently:

    - **Confirming the order**: Unconfirmed orders can be canceled. This usually takes 10-60 seconds.
    - **Preparing the order**: Baking a pizza typically takes around 20 minutes, but can take longer if a mistake is made (e.g., baking the wrong pizza).
    - **Delivering the order**: Delivery time depends on weather, traffic, and distance. For example, a 10 km distance typically takes 15 minutes, but heavy traffic can extend this to nearly an hour.

    To optimize time, multitask where possible:
    - Bake multiple pizzas if several orders come in simultaneously.
    - Confirm new orders or start preparing a new pizza while waiting for another to finish.
    - Deliver multiple orders in one trip if they are nearby.

    However, note:
    - You cannot bake while driving.
    - Prioritize tasks based on order status and time efficiency, ensuring customers don't wait too long and that pizzas arrive hot.

    **Actions you can take**:
    - Confirm order
    - Start preparing order
    - Start delivering order
    - Finish delivering order
    - Cancel order (only in special circumstances)
    - Do nothing (if there's nothing to do, or you can't do anything right now)

    You can start preparing an order right after confirming it, as confirmation is quick. However, you can't deliver until the pizza is baked, you can't bake while delivering because you're in your car. 

    Important: 
    - Always call tools to check traffic and weather conditions carefully because your action depends also heavily on the weather and the traffic. Poor waether or/and traffic conditions delay deliveries and you should prioritize closer orders, or consider canceling if fulfillment is unrealistic. \
    - You only work until 16:00, maximum 18:00, you should cancel orders which you can't fulfil before 18:00. 
    - You should not take orders and let customer waits for more than 3 hours. If an order may takes too long, you should consider cancelling it. And due to weather and traffic condition, deliver an order can takes very long.
    - Either you start delivering many orders at the same time, otherwise you have to finish delivering (order delivered) before doing anything (unless you're confirming an order during the delivery process). 
    - You should not let pizzas being baked and start delivering. That means, make sure you start delivering without any order being prepared.
    - Even you start delivering many orders at the same time, you can't deliver them at the same time because they are different orders from different customers living in different places. The time each order takes depends on the distance and the time you start delivering.

    Below is the latest states of all orders:
    {latest_state}

    Based on weather and traffic conditions, orders's latest states, decide on the next action(s) within the next 10 minutes, from {start_time} to {end_time}. Provide the following:
    - **order_id**: The order being processed
    - **action**: The action you're taking ("confirm order", "start preparing order", "start delivering order", "finish delivering order", "cancel order", or "do nothing")
    - **timestamp**: The time at which you take the action (within the time frame)

    If handling multiple orders, provide each action separately in the same format. 
    """
    latest_state = get_latest_states(system_state)
    prompt = pizzeria_prompt.format(latest_state=latest_state, start_time=start_time, end_time=end_time)
    # initializaing messages with human prompt
    
    messages = [HumanMessage(prompt)]

    if call_tools:
        # Step 1: Use LLM with tools to determine if tool calls are needed
        llm_with_tools = llm.bind_tools([GetWeather, GetTraffic])
        ai_response = llm_with_tools.invoke(messages)

        if ai_response.tool_calls:
            print('Pizzeria calling tools')

            if ai_response.content:
                messages.append(ai_response)

            # Step 2: Execute each tool and append its response
            tool_map = {'getweather': get_weather, 'gettraffic': get_traffic}
            for tool_call in ai_response.tool_calls:
                tool_fn = tool_map[tool_call['name'].lower()]
                tool_result = tool_fn.invoke(tool_call['args'])
                if tool_result:
                    messages.append(ToolMessage(tool_result, tool_call_id=tool_call['id']))
                else:
                    continue

            # Step 3: Re-invoke LLM with all messages to generate final response
            final_response = llm_with_tools.invoke(messages)
            if final_response.content:
                messages.append(final_response)
            else:
                print('Unstructured output is empty')

        else:
            print('Pizzeria does not call tools')
            ai_response = llm.invoke(messages)
            if ai_response.content:
                messages.append(ai_response)
    else:
        # No tools used, simple LLM call
        ai_response = llm.invoke(messages)
        if ai_response.content:
            messages.append(ai_response)

    # Step 4: Extract structured output from full conversation
    structured_llm = llm.with_structured_output(PizzeriaOutputMultiple)
    output = structured_llm.invoke(messages)

    return output


### Updating functions

In [50]:
from typing import TypedDict, List, Dict

class Orderstate(TypedDict):
    customer_name: str
    distance: str
    pizza_type: str
    states: List[str]
    timestamps: List[str]

class CustomerInfo(TypedDict):
    name: str
    character: str
    distance: str

In [51]:
import copy

act2state_dict = {
    'place order': 'order placed',
    'confirm order': 'order confirmed',
    'cancel order': 'order cancelled',
    'start preparing order': 'order being prepared',
    'start delivering order': 'order being delivered',
    'finish delivering order': 'order delivered',
    'wait for order': '',
    'do nothing': ''
}
def act2state_map(action):
    state = act2state_dict.get(action)
    if state is None:
        state = ''
        raise KeyError(f"Action '{action}' is not in the predefined action list.")
    return state

def customer_order_update(all_orders_dict, un_orders_dict, id2customer_dict, customer, order_id, output):
    if order_id in all_orders_dict:
        raise ValueError(f"Order ID {order_id} already exists. It must be unique.")

    # Create a new order state
    order_state = Orderstate(
        customer_name=customer['name'],
        distance=customer['distance'],
        pizza_type=output.pizza_type,
        states=[act2state_map(output.action)],  
        timestamps=[output.timestamp]  
    )

    # Add order to all orders
    all_orders_dict[order_id] = order_state

    # Add to unfinished orders  
    un_orders_dict[order_id] = copy.deepcopy(order_state)

    # Update order-to-customer mapping
    id2customer_dict[order_id] = customer

def customer_cancel_update(all_orders_dict, un_orders_dict, order_id, output):
    state = act2state_map(output.action.lower())
    timestamp = output.timestamp

    # Update all orders
    if state:
        if order_id in all_orders_dict:
            all_orders_dict[order_id]['states'].append(state)
            all_orders_dict[order_id]['timestamps'].append(timestamp)
        else:
            print(f"Order ID {order_id} not found in all_orders_dict.")

        # Update unfinished orders only if the order is still active
        if order_id in un_orders_dict:
            if state in ["order cancelled"]:
                # Remove directly without appending redundant data
                del un_orders_dict[order_id]
            else:
                # Only update if the order is still active
                un_orders_dict[order_id]['states'].append(state)
                un_orders_dict[order_id]['timestamps'].append(timestamp)

def pizzeria_action_update(all_orders_dict, un_orders_dict, outputs):
    for output in outputs.multiple_actions:
        order_id = output.order_id
        state = act2state_map(output.action.lower())
        timestamp = output.timestamp

        # Update all orders
        if state:
            if order_id in all_orders_dict:
                all_orders_dict[order_id]['states'].append(state)
                all_orders_dict[order_id]['timestamps'].append(timestamp)
            else:
                print(f"Order ID {order_id} not found in all_orders_dict.")
                continue

            # Update unfinished orders only if the order is still active
            if order_id in un_orders_dict:
                if state in ["order cancelled", "order delivered"]:
                    # Remove directly without appending redundant data
                    del un_orders_dict[order_id]
                else:
                    # Only update if the order is still active
                    un_orders_dict[order_id]['states'].append(state)
                    un_orders_dict[order_id]['timestamps'].append(timestamp)

def check_unconfirmed_order(un_orders_dict):
    un_confirmed_orders = []
    for order_if, order_state in un_orders_dict.items():
        if "order placed" in order_state['states'] and "order confirmed" not in order_state['states']:
            un_confirmed_orders.append(order_if)
    return un_confirmed_orders



### Running simulation

In [52]:
# initialize order process state variable
all_orders_dict: Dict[str, Orderstate] = {}  # all orders' states 
un_orders_dict: Dict[str, Orderstate] = {} # uncompleted orders' states
id2customer_dict: Dict[str, CustomerInfo] = {}  # Key: order ID (str), Value: CustomerInfo dict # mapping order vs customer

called_customers = []
order_counter = 0

In [53]:
from datetime import datetime, timedelta

num_interation = 26 #(from 12:00 to 16:00)
start_time = datetime.fromisoformat("2025-04-04T12:00:00")
for i in range(num_interation):
    print(f'Start_time: {start_time} and num of iteration {i}')
    # First, we need to generate a number of customer
    from helpers import generate_num_orders
    num_orders = generate_num_orders(start_time)

    # Then, we call on these customers to see if they would love to order pizza(s) or not
    from helpers import generate_unique_customer
    from helpers import generate_order_id

    end_time = start_time + timedelta(minutes=10)

    for j in range(num_orders):
        # generate random customer
        customer = generate_unique_customer(called_customers)
        # call on customer ordering agent
        output = customer_order_agent(customer=customer, start_time=start_time, end_time=end_time, call_tools=True)
        if output.action == "place order":
            order_id = generate_order_id(order_counter)
            order_counter += 1
            # update order process states
            customer_order_update(all_orders_dict=all_orders_dict,
                                un_orders_dict=un_orders_dict,
                                id2customer_dict=id2customer_dict,
                                customer=customer,
                                order_id=order_id,
                                output=output)
            
    # Next, we call all pizzeria agent
    if all_orders_dict:
        outputs = pizzeria_agent(system_state=un_orders_dict, start_time=start_time, end_time=end_time, call_tools=True)
        # Update order process state
        pizzeria_action_update(all_orders_dict=all_orders_dict,
                            un_orders_dict=un_orders_dict,
                            outputs=outputs)
        # Now we need to check if there is any order placed but not confirmed, we will call on the customer cancel agent
    un_confirmed_orders = check_unconfirmed_order(un_orders_dict=un_orders_dict)
    if un_confirmed_orders:
        for un_con_id in un_confirmed_orders:
            customer = id2customer_dict[un_con_id]
            order_info = un_orders_dict[un_con_id]
            # call on customer cancel agent
            output = customer_cancel_agent(customer=customer, order=order_info, start_time=start_time, end_time=end_time)
            # update order process state
            customer_cancel_update(all_orders_dict=all_orders_dict,
                                un_orders_dict=un_orders_dict,
                                order_id=un_con_id,
                                output=output)

    # updating time
    print(f'End_time: {end_time}')
    start_time = end_time + timedelta(seconds=30)


Start_time: 2025-04-04 12:00:00 and num of iteration 0
Oder customer calling tools
Pizzeria calling tools
End_time: 2025-04-04 12:10:00
Start_time: 2025-04-04 12:10:30 and num of iteration 1
Oder customer calling tools
Pizzeria calling tools
End_time: 2025-04-04 12:20:30
Start_time: 2025-04-04 12:21:00 and num of iteration 2
Oder customer calling tools
Oder customer calling tools
Oder customer calling tools
Pizzeria does not call tools
End_time: 2025-04-04 12:31:00
Start_time: 2025-04-04 12:31:30 and num of iteration 3
Oder customer calling tools
Oder customer calling tools
Oder customer calling tools
Pizzeria calling tools
Unstructured output is empty
End_time: 2025-04-04 12:41:30
Start_time: 2025-04-04 12:42:00 and num of iteration 4
Pizzeria calling tools
Unstructured output is empty
End_time: 2025-04-04 12:52:00
Start_time: 2025-04-04 12:52:30 and num of iteration 5
Oder customer calling tools
Oder customer calling tools
Pizzeria calling tools
End_time: 2025-04-04 13:02:30
Start_ti

### Event Log DF

In [12]:
len(un_orders_dict)

15

In [54]:
import pandas as pd
def convert_to_dataframe(all_orders_dict: Dict[str, Orderstate]) -> pd.DataFrame:
    order_data = []
    
    # Iterate through all orders
    for order_id, order in all_orders_dict.items():
        states = order['states']
        timestamps = order['timestamps']
        
        # Check if states and timestamps have matching lengths
        if len(states) != len(timestamps):
            raise ValueError(f"Mismatch between number of states and timestamps for order {order_id}")
        
        # Add each state-timestamp pair to the order_data list
        for state, timestamp in zip(states, timestamps):
            order_data.append({
                'order_id': order_id,
                'state': state,
                'timestamp': timestamp
            })
    
    # Convert to DataFrame
    df = pd.DataFrame(order_data)
    return df

df = convert_to_dataframe(all_orders_dict)
df_sorted = df.sort_values(by=['timestamp'])
# Save to CSV
df.to_csv('orders_tool_calls_1.csv', index=False)


In [43]:
order_counter

71

In [ ]:
### Event Log Analysis